# Métodos Exatos
---

In [40]:
import sys
from pathlib import Path

diretorio_atual = Path.cwd()

pasta_src = diretorio_atual / "article" / "src"

if str(pasta_src) not in sys.path:
    sys.path.append(str(pasta_src))

from metamodel import MetaModel

In [41]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

from pulp import *
import warnings

In [42]:
def save_csv(data: pd.DataFrame, name: str = "test") -> bool:
    
    path = Path("results/abordagens_metodos_exatos")
    path.mkdir(exist_ok=True)
    
    try:
        data.to_csv(str(path) + f"/{name}.csv", index=True)
        return True
    except Exception as e:
        print("[Erro CSV]: ", e)
        return False

In [43]:
meta_dataset = pd.read_csv('data/metafeatures_dataset_with_best.csv', index_col = 0)

In [44]:
meta_models = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "naive_bayes": make_pipeline(StandardScaler(), GaussianNB()),
    "svm": make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=42)),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
}

In [45]:
def execute_meta_models(meta_dataset, meta_dataset_train, alg = "test", method = "test"):
    
    if len(meta_dataset_train.columns) <= 2:
        return [0, alg, 0.0, 0.0, 0.0, 0.0, 0.0, 'None', method]
    
    row = [len(meta_dataset_train.columns), alg]
    best = [0.0, 'nenhum']

    for key, value in meta_models.items():
        
        metamodel = MetaModel()
        predicts = metamodel.train_and_evaluate_best_metamodel(meta_dataset_train=meta_dataset_train,
                                                            meta_dataset=meta_dataset,
                                                            model=value)
        y_pred = list(predicts['Best clf (pred)'])
        y_true = meta_dataset['Best'].values
        
        metamodel_accuracy = accuracy_score(y_true, y_pred)
        metamodel_f1 = f1_score(y_true, y_pred, average='weighted')
        
        print(f"### {key} ###")
        print(f'Meta-model Accuracy: {metamodel_accuracy:.2f}')
        print(f'Meta-model F1-score: {metamodel_f1:.2f}')
        print(f"####{len(key)*'#'}####")
        
        if best[0] <= metamodel_accuracy:
            best[0] = metamodel_accuracy
            best[1] = key
        
        row.append(metamodel_accuracy)
        del metamodel
            
    row.append(best[1])
    row.append(method)
    
    return row
    

In [46]:
df_results = pd.DataFrame({
    "n_features": [],
    "alg_selected_meta_features": [],
    "decision_tree": [],
    "random_forest": [],
    "naive_bayes": [],
    "svm": [],
    "knn": [],
    "best": [],
    "method": []
})

df_results.loc[len(df_results)] = execute_meta_models(meta_dataset=meta_dataset,
                                                        meta_dataset_train=meta_dataset,
                                                        alg = "all meta features",
                                                        method = "all meta features")


### decision_tree ###
Meta-model Accuracy: 0.43
Meta-model F1-score: 0.43
#####################
### random_forest ###
Meta-model Accuracy: 0.53
Meta-model F1-score: 0.49
#####################
### naive_bayes ###
Meta-model Accuracy: 0.43
Meta-model F1-score: 0.39
###################
### svm ###
Meta-model Accuracy: 0.54
Meta-model F1-score: 0.48
###########
### knn ###
Meta-model Accuracy: 0.39
Meta-model F1-score: 0.37
###########


In [47]:
df_results.head(10)

,n_features,alg_selected_meta_features,decision_tree,random_forest,naive_bayes,svm,knn,best,method
0,1154,all meta features,0.425532,0.531915,0.425532,0.542553,0.393617,svm,all meta features


In [48]:
meta_model_feature_scores = {}

for key, model in meta_models.items():
    meta_model = MetaModel()
    feature_scores = meta_model.meta_feature_scores(meta_dataset=meta_dataset, model=model)
    meta_model_feature_scores[key] = feature_scores


## Best Subset Selection via MILP
---

Programação Linear Inteira Mista (MILP)

In [49]:
def best_subset_selection_filter_milp(meta_dataset, max_feature: int = 10):
    warnings.filterwarnings("ignore")
    
    meta_features = [col for col in meta_dataset.columns if col != 'Best' and col != 'Dataset']
    selected_meta_features_by_algorithm = {}
    
    for key, _ in meta_models.items():
        
        problem = LpProblem(f"Best_Subset_Filter_{key}", LpMaximize)
        
        x = {i: LpVariable(f"{meta_features[i]}", cat="Binary") for i in range(len(meta_features))}
        
        problem += lpSum(meta_model_feature_scores[key][i] * x[i] for i in range(len(meta_features)))
        
        problem += lpSum(x[i] for i in range(len(meta_features))) == max_feature
        
        problem.solve(PULP_CBC_CMD(msg=False))
        
        selected_meta_features = [meta_features[i] for i in range(len(meta_features)) if x[i].varValue == 1.0]
        selected_meta_features_by_algorithm[key] = selected_meta_features
        
    return selected_meta_features_by_algorithm

In [50]:
selected_meta_features = best_subset_selection_filter_milp(meta_dataset=meta_dataset,
                             max_feature=200)

print("Selecionando as meta_features para cada algoritmo com Best Subset Selection!!!")

Selecionando as meta_features para cada algoritmo com Best Subset Selection!!!


In [51]:
for model, meta_features in selected_meta_features.items():
    
    print(f"Features selecionadas para modelo: {model}")
    print("Número de meta features selecionadas: ", len(meta_features))
    
    meta_dataset_train = meta_dataset[meta_features + ['Dataset', 'Best']]
    
    df_results.loc[len(df_results)] = execute_meta_models(meta_dataset=meta_dataset,
                                                        meta_dataset_train=meta_dataset_train,
                                                        alg = model,
                                                        method = "Best Subset Selection by Filter MILP")

Features selecionadas para modelo: decision_tree
Número de meta features selecionadas:  200
### decision_tree ###
Meta-model Accuracy: 0.63
Meta-model F1-score: 0.63
#####################
### random_forest ###
Meta-model Accuracy: 0.47
Meta-model F1-score: 0.42
#####################
### naive_bayes ###
Meta-model Accuracy: 0.35
Meta-model F1-score: 0.35
###################
### svm ###
Meta-model Accuracy: 0.45
Meta-model F1-score: 0.36
###########
### knn ###
Meta-model Accuracy: 0.44
Meta-model F1-score: 0.42
###########
Features selecionadas para modelo: random_forest
Número de meta features selecionadas:  200
### decision_tree ###
Meta-model Accuracy: 0.44
Meta-model F1-score: 0.43
#####################
### random_forest ###
Meta-model Accuracy: 0.51
Meta-model F1-score: 0.48
#####################
### naive_bayes ###
Meta-model Accuracy: 0.43
Meta-model F1-score: 0.43
###################
### svm ###
Meta-model Accuracy: 0.54
Meta-model F1-score: 0.47
###########
### knn ###
Meta-mod

In [52]:
df_results.head(40)

,n_features,alg_selected_meta_features,decision_tree,random_forest,naive_bayes,svm,knn,best,method
0,1154,all meta features,0.425532,0.531915,0.425532,0.542553,0.393617,svm,all meta features
1,202,decision_tree,0.627660,0.468085,0.351064,0.446809,0.436170,decision_tree,Best Subset Selection by Filter MILP
2,202,random_forest,0.436170,0.510638,0.425532,0.542553,0.404255,svm,Best Subset Selection by Filter MILP
3,146,naive_bayes,0.382979,0.382979,0.234043,0.382979,0.361702,svm,Best Subset Selection by Filter MILP
4,202,svm,0.276596,0.425532,0.340426,0.457447,0.382979,svm,Best Subset Selection by Filter MILP
5,202,knn,0.212766,0.468085,0.414894,0.489362,0.351064,svm,Best Subset Selection by Filter MILP


In [53]:
if save_csv(data=df_results, name="metafeatures_dataset_with_best"):
    print("OK! Csv criado com sucesso!")
else:
    print("Erro ao tentar salvar csv!")

OK! Csv criado com sucesso!


## Filtro Otimizado via Programação Linear Inteira
---

Adaptação do `problema da mochila`

A partir de um determinado limite `max_feature` pegue a melhor combinção possível

In [54]:
warnings.filterwarnings("ignore")

def knapsack_problem_ilp(meta_dataset, max_feature: int = 10, l: float = 0.01):
    
    meta_features = [col for col in meta_dataset.columns if col != 'Best' and col != 'Dataset']
    selected_meta_features_by_algorithm = {}
    
    for key, model in meta_models.items():

        problem = LpProblem("Meta_Feature_Selection", LpMaximize)

        x = {i: LpVariable(f"{meta_features[i]}", cat="Binary") for i in range(len(meta_features))}

        # Hiperparâmetros
        MAX_FEATURES = max_feature  # Número máximo de meta-features que você deseja
        LAMBDA = l     # Penalidade para features com importância muito baixa

        problem += lpSum((meta_model_feature_scores[key][i] - LAMBDA) * x[i] for i in range(len(meta_features)))
        problem += lpSum(x[i] for i in range(len(meta_features))) <= MAX_FEATURES

        problem.solve(PULP_CBC_CMD(msg=False))
        
        selected_meta_features = [meta_features[i] for i in range(len(meta_features)) if x[i].varValue == 1.0]
        selected_meta_features_by_algorithm[key] = selected_meta_features
        
    return selected_meta_features_by_algorithm


In [55]:
selected_meta_features = knapsack_problem_ilp(meta_dataset=meta_dataset,
                             max_feature=200,
                             l=0.002)

print("Selecionando as meta_features para cada algoritmo com Filtro Otimizado!!!")

Selecionando as meta_features para cada algoritmo com Filtro Otimizado!!!


In [56]:
for model, meta_features in selected_meta_features.items():
    
    print("Algoritmo responsável: ", model)
    print("Número de meta features selecionadas: ", len(meta_features))
    
    meta_dataset_train = meta_dataset[meta_features + ['Dataset', 'Best']]
    
    df_results.loc[len(df_results)] = execute_meta_models(meta_dataset=meta_dataset,
                                                        meta_dataset_train=meta_dataset_train,
                                                        alg = model,
                                                        method = "Knapsack Problem ILP")
    

Algoritmo responsável:  decision_tree
Número de meta features selecionadas:  23
### decision_tree ###
Meta-model Accuracy: 0.68
Meta-model F1-score: 0.68
#####################
### random_forest ###
Meta-model Accuracy: 0.51
Meta-model F1-score: 0.47
#####################
### naive_bayes ###
Meta-model Accuracy: 0.17
Meta-model F1-score: 0.15
###################
### svm ###
Meta-model Accuracy: 0.44
Meta-model F1-score: 0.36
###########
### knn ###
Meta-model Accuracy: 0.34
Meta-model F1-score: 0.32
###########
Algoritmo responsável:  random_forest
Número de meta features selecionadas:  147
### decision_tree ###
Meta-model Accuracy: 0.40
Meta-model F1-score: 0.40
#####################
### random_forest ###
Meta-model Accuracy: 0.53
Meta-model F1-score: 0.51
#####################
### naive_bayes ###
Meta-model Accuracy: 0.38
Meta-model F1-score: 0.40
###################
### svm ###
Meta-model Accuracy: 0.54
Meta-model F1-score: 0.47
###########
### knn ###
Meta-model Accuracy: 0.44
Meta-

In [57]:
df_results.head(40)

,n_features,alg_selected_meta_features,decision_tree,random_forest,naive_bayes,svm,knn,best,method
0,1154,all meta features,0.425532,0.531915,0.425532,0.542553,0.393617,svm,all meta features
1,202,decision_tree,0.627660,0.468085,0.351064,0.446809,0.436170,decision_tree,Best Subset Selection by Filter MILP
2,202,random_forest,0.436170,0.510638,0.425532,0.542553,0.404255,svm,Best Subset Selection by Filter MILP
3,146,naive_bayes,0.382979,0.382979,0.234043,0.382979,0.361702,svm,Best Subset Selection by Filter MILP
4,202,svm,0.276596,0.425532,0.340426,0.457447,0.382979,svm,Best Subset Selection by Filter MILP
5,202,knn,0.212766,0.468085,0.414894,0.489362,0.351064,svm,Best Subset Selection by Filter MILP
6,25,decision_tree,0.680851,0.510638,0.170213,0.436170,0.340426,decision_tree,Knapsack Problem ILP
7,149,random_forest,0.404255,0.531915,0.382979,0.542553,0.436170,svm,Knapsack Problem ILP
8,202,naive_bayes,0.361702,0.521277,0.414894,0.574468,0.563830,svm,Knapsack Problem ILP
9,52,svm,0.287234,0.425532,0.287234,0.372340,0.361702,random_forest,Knapsack Problem ILP


In [58]:
if save_csv(data=df_results, name="metafeatures_dataset_with_best"):
    print("OK! Csv criado com sucesso!")
else:
    print("Erro ao tentar salvar csv!")

OK! Csv criado com sucesso!


## PLI com Penalização de Correlação

---

In [59]:
def correlation_penalized_by_milp(meta_dataset, max_feature: int = 10, gama: float = 0.05, corr_threshold: float = 0.3):
    warnings.filterwarnings("ignore")
    
    meta_features = [col for col in meta_dataset.columns if col != 'Best' and col != 'Dataset']
    selected_meta_features_by_algorithm = {}
    
    corr_matrix = meta_dataset[meta_features].corr().abs().fillna(0).values
    
    for key, _ in meta_models.items():
        print(f"-> Otimizando subconjunto para o modelo: {key}...")
        
        problem = LpProblem(f"Penalizacao_Corr_{key}", LpMaximize)
        
        x = {i: LpVariable(f"x_{i}", cat="Binary") for i in range(len(meta_features))}

        z = {}
        for i in range(len(meta_features)):
            for j in range(i + 1, len(meta_features)):
                if corr_matrix[i][j] >= corr_threshold:
                    z[(i, j)] = LpVariable(f"z_{i}_{j}", cat="Binary")

        for (i, j) in z:
            problem += z[i, j] <= x[i]
            problem += z[i, j] <= x[j]
            problem += z[i, j] >= x[i] + x[j] - 1

        problem += lpSum(x[i] for i in range(len(meta_features))) <= max_feature

        linear_scores = lpSum(meta_model_feature_scores[key][i] * x[i] for i in range(len(meta_features)))
        redundancy_penalty = lpSum(corr_matrix[i][j] * z[i, j] for (i, j) in z)

        problem += linear_scores - (gama * redundancy_penalty)
        
        solver = PULP_CBC_CMD(msg=False, timeLimit=45, gapRel=0.02)
        problem.solve(solver)
        
        selected_meta_features = [meta_features[i] for i in range(len(meta_features)) if x[i].varValue == 1.0]
        selected_meta_features_by_algorithm[key] = selected_meta_features
        print(f"   Concluído! Selecionadas {len(selected_meta_features)} features.")
    
    return selected_meta_features_by_algorithm

In [60]:
selected_meta_features = correlation_penalized_by_milp(meta_dataset=meta_dataset,
                                                       max_feature=200)

print("Selecionando as meta_features para cada algoritmo Correlação Penalizada com MILP!!!")

-> Otimizando subconjunto para o modelo: decision_tree...
   Concluído! Selecionadas 16 features.
-> Otimizando subconjunto para o modelo: random_forest...
   Concluído! Selecionadas 111 features.
-> Otimizando subconjunto para o modelo: naive_bayes...
   Concluído! Selecionadas 0 features.
-> Otimizando subconjunto para o modelo: svm...
   Concluído! Selecionadas 20 features.
-> Otimizando subconjunto para o modelo: knn...
   Concluído! Selecionadas 71 features.
Selecionando as meta_features para cada algoritmo Correlação Penalizada com MILP!!!


In [61]:
for model, meta_features in selected_meta_features.items():
    
    print(f"Features selecionadas para modelo: {model}")
    print(f"Número de features: ", len(meta_features))
    
    meta_dataset_train = meta_dataset[meta_features + ['Dataset', 'Best']]
    
    df_results.loc[len(df_results)] = execute_meta_models(meta_dataset=meta_dataset,
                                                        meta_dataset_train=meta_dataset_train,
                                                        alg = model,
                                                        method = "Correlation Penalized milp")

Features selecionadas para modelo: decision_tree
Número de features:  16
### decision_tree ###
Meta-model Accuracy: 0.67
Meta-model F1-score: 0.67
#####################
### random_forest ###
Meta-model Accuracy: 0.56
Meta-model F1-score: 0.54
#####################
### naive_bayes ###
Meta-model Accuracy: 0.15
Meta-model F1-score: 0.14
###################
### svm ###
Meta-model Accuracy: 0.43
Meta-model F1-score: 0.30
###########
### knn ###
Meta-model Accuracy: 0.33
Meta-model F1-score: 0.27
###########
Features selecionadas para modelo: random_forest
Número de features:  111
### decision_tree ###
Meta-model Accuracy: 0.44
Meta-model F1-score: 0.43
#####################
### random_forest ###
Meta-model Accuracy: 0.50
Meta-model F1-score: 0.44
#####################
### naive_bayes ###
Meta-model Accuracy: 0.39
Meta-model F1-score: 0.41
###################
### svm ###
Meta-model Accuracy: 0.47
Meta-model F1-score: 0.40
###########
### knn ###
Meta-model Accuracy: 0.41
Meta-model F1-score

In [62]:
df_results.head(40)

,n_features,alg_selected_meta_features,decision_tree,random_forest,naive_bayes,svm,knn,best,method
0,1154,all meta features,0.425532,0.531915,0.425532,0.542553,0.393617,svm,all meta features
1,202,decision_tree,0.627660,0.468085,0.351064,0.446809,0.436170,decision_tree,Best Subset Selection by Filter MILP
2,202,random_forest,0.436170,0.510638,0.425532,0.542553,0.404255,svm,Best Subset Selection by Filter MILP
3,146,naive_bayes,0.382979,0.382979,0.234043,0.382979,0.361702,svm,Best Subset Selection by Filter MILP
4,202,svm,0.276596,0.425532,0.340426,0.457447,0.382979,svm,Best Subset Selection by Filter MILP
5,202,knn,0.212766,0.468085,0.414894,0.489362,0.351064,svm,Best Subset Selection by Filter MILP
6,25,decision_tree,0.680851,0.510638,0.170213,0.436170,0.340426,decision_tree,Knapsack Problem ILP
7,149,random_forest,0.404255,0.531915,0.382979,0.542553,0.436170,svm,Knapsack Problem ILP
8,202,naive_bayes,0.361702,0.521277,0.414894,0.574468,0.563830,svm,Knapsack Problem ILP
9,52,svm,0.287234,0.425532,0.287234,0.372340,0.361702,random_forest,Knapsack Problem ILP


In [63]:
if save_csv(data=df_results, name="metafeatures_dataset_with_best"):
    print("OK! Csv criado com sucesso!")
else:
    print("Erro ao tentar salvar csv!")

OK! Csv criado com sucesso!
